# Proyecto 2 - Detección de objetos con YOLO
## Detección de residuos reciclables

**Integrantes:**  Juan José Betancourt Osorio, Jean Pierre Henriquez Ebrat, Guillermo Alexander Zabala Fernandez
**Curso:** Visión Computacional con Deep Learning  
**Modelo base:** YOLO11s  
**Dataset:** residuos_dataset

### Objetivo
Entrenar un modelo YOLO con transferencia de aprendizaje para detectar tipos de residuos en imágenes y evaluar su desempeño en datos no vistos.


In [ ]:
%pip install -q ultralytics opencv-python matplotlib pyyaml pandas pillow

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import yaml
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import random
import os


In [ ]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATASET_DIR = PROJECT_ROOT / "data" / "residuos_dataset"
DATA_YAML = DATASET_DIR / "data.yaml"
RUNS_DIR = PROJECT_ROOT / "runs"


print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_DIR:", DATASET_DIR)
print("DATA_YAML:", DATA_YAML)
print("Existe data.yaml:", DATA_YAML.exists())
print("CUDA disponible:", torch.cuda.is_available())


In [ ]:
with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

data_cfg


In [ ]:
required_paths = [
    DATASET_DIR / "train" / "images",
    DATASET_DIR / "train" / "labels",
    DATASET_DIR / "valid" / "images",
    DATASET_DIR / "valid" / "labels",
    DATASET_DIR / "test" / "images",
    DATASET_DIR / "test" / "labels",
]

for p in required_paths:
    print(p, "->", p.exists())


In [ ]:
def count_images(folder):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return len([f for f in folder.iterdir() if f.suffix.lower() in exts])

train_count = count_images(DATASET_DIR / "train" / "images")
valid_count = count_images(DATASET_DIR / "valid" / "images")
test_count = count_images(DATASET_DIR / "test" / "images")

print("Train:", train_count)
print("Valid:", valid_count)
print("Test:", test_count)
print("Total:", train_count + valid_count + test_count)


In [ ]:
train_images_dir = DATASET_DIR / "train" / "images"
image_files = list(train_images_dir.glob("*.*"))
sample_files = random.sample(image_files, min(6, len(image_files)))

plt.figure(figsize=(14, 8))
for i, img_path in enumerate(sample_files, 1):
    img = Image.open(img_path)
    plt.subplot(2, 3, i)
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
model = YOLO("yolo11s.pt")
model.info()


In [ ]:
model = YOLO("yolo11s.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=30,
    imgsz=640,
    batch=8,
    pretrained=True,
    project=str(RUNS_DIR),
    name="residuos_yolo11s",
    val=True,
    plots=True,
    patience=15,
    device=0 if torch.cuda.is_available() else "cpu"
)


In [ ]:
best_model_path = RUNS_DIR / "residuos_yolo11s" / "weights" / "best.pt"
last_model_path = RUNS_DIR / "residuos_yolo11s" / "weights" / "last.pt"

print("best.pt:", best_model_path, best_model_path.exists())
print("last.pt:", last_model_path, last_model_path.exists())


In [ ]:
import shutil

final_model_path = MODELS_DIR / "best_residuos_yolo11s.pt"

if best_model_path.exists():
    shutil.copy2(best_model_path, final_model_path)
    print("Modelo copiado a:", final_model_path)
else:
    print("No se encontró best.pt")


In [ ]:
best_model = YOLO(str(final_model_path))
best_model.info()


In [ ]:
metrics = best_model.val(data=str(DATA_YAML))
metrics


In [ ]:
results_csv = RUNS_DIR / "residuos_yolo11s" / "results.csv"
df = pd.read_csv(results_csv)
df.head()


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df["epoch"], df["train/box_loss"], label="train box loss")
plt.plot(df["epoch"], df["val/box_loss"], label="val box loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Pérdida de entrenamiento y validación")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df["epoch"], df["metrics/precision(B)"], label="Precision")
plt.plot(df["epoch"], df["metrics/recall(B)"], label="Recall")
plt.plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50")
plt.plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95")
plt.xlabel("Epoch")
plt.ylabel("Valor")
plt.title("Métricas del modelo")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
test_images = list((DATASET_DIR / "test" / "images").glob("*.*"))
sample_test_image = test_images[0]

print("Imagen de prueba:", sample_test_image)

pred = best_model.predict(
    source=str(sample_test_image),
    conf=0.25,
    save=True,
    project=str(RUNS_DIR),
    name="predict_test_image",
    exist_ok=True
)


In [ ]:
predicted_img_path = RUNS_DIR / "predict_test_image" / sample_test_image.name
img = Image.open(predicted_img_path)

plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis("off")
plt.show()


In [ ]:
predictions = best_model.predict(source=str(sample_test_image), conf=0.25)
class_names = best_model.names

for pred in predictions:
    for box, conf, cls in zip(pred.boxes.xyxy, pred.boxes.conf, pred.boxes.cls):
        print(
            "Clase:", class_names[int(cls.item())],
            "| Confianza:", round(conf.item(), 4),
            "| Box:", [round(x, 2) for x in box.tolist()]
        )


In [ ]:
video_path = PROJECT_ROOT / "test_video.mp4"

if video_path.exists():
    best_model.predict(
        source=str(video_path),
        conf=0.25,
        save=True,
        project=str(RUNS_DIR),
        name="predict_video",
        exist_ok=True
    )
    print("Video procesado.")
else:
    print("No existe test_video.mp4 en la raíz del proyecto.")


In [ ]:
# Ejecutar esta celda solo para probar con camara
# best_model.predict(source=0, show=True, conf=0.25)
